In [20]:
# Instalação das dependências necessárias (caso rodado no Google Colab)
#!pip install pandasql
#!pip install pandas
#%pip install pandas pandasql

In [21]:
import pandas as pd
from pandasql import sqldf
import numpy as np

In [22]:
# Função helper para executar queries no escopo global usando a sintaxe SQLite
pysqldf = lambda q: sqldf(q, globals())

In [23]:
# URLs Raw do repositório no GitHub para reprodutibilidade total
BASE_URL = "https://raw.githubusercontent.com/lkalilduarte/desafio-sql-marketplace/main/dados/"

buyers = pd.read_csv(f"{BASE_URL}buyers.csv")
order_items = pd.read_csv(f"{BASE_URL}order_items.csv")
orders = pd.read_csv(f"{BASE_URL}orders.csv")
payments = pd.read_csv(f"{BASE_URL}payments.csv")
products = pd.read_csv(f"{BASE_URL}products.csv")
sellers = pd.read_csv(f"{BASE_URL}sellers.csv")

In [24]:
# Mapeamento das tabelas e suas Chaves Primárias (PK)
tabelas = {
    'buyers': {'file': 'buyers.csv', 'pk': 'id'},
    'order_items': {'file': 'order_items.csv', 'pk': 'id'},
    'orders': {'file': 'orders.csv', 'pk': 'id'},
    'payments': {'file': 'payments.csv', 'pk': 'id'},
    'products': {'file': 'products.csv', 'pk': 'id'},
    'sellers': {'file': 'sellers.csv', 'pk': 'id'}
}

for nome, info in tabelas.items():
    url = f"{BASE_URL}{info['file']}"
    df = pd.read_csv(url)
    pk = info['pk']
    
    print(f"\n TABELA: [{nome.upper()}] | Linhas: {df.shape[0]} | Colunas: {df.shape[1]}")
    
    # 1. Checagem de Chave Primária e Duplicadas Totais
    duplicados_totais = df.duplicated().sum()
    duplicados_pk = df[pk].duplicated().sum() if pk in df.columns else 0
    
    if duplicados_totais > 0:
        print(f"  Linhas totalmente duplicadas: {duplicados_totais}")
    if duplicados_pk > 0:
        print(f"  Chaves Primárias ('{pk}') duplicadas: {duplicados_pk}")
    if duplicados_totais == 0 and duplicados_pk == 0:
        print(f"  Sem duplicidades de registros ou de Chave Primária ('{pk}').")
        
    # 2. Checagem de Valores Nulos e Espaços em Branco
    anomalias_colunas = []
    
    for col in df.columns:
        nulos = df[col].isnull().sum()
        pct_nulos = (nulos / len(df)) * 100
        
        # Checagem de texto vazio/espaço em branco em colunas do tipo object/string
        vazios = 0
        if df[col].dtype == 'object':
            vazios = df[col].astype(str).str.strip().eq('').sum()
            
        if nulos > 0 or vazios > 0:
            anomalias_colunas.append({
                'Coluna': col,
                'Nulos': nulos,
                '% Nulos': f"{pct_nulos:.2f}%",
                'Vazios/Brancos': vazios
            })
            
    if anomalias_colunas:
        print("\n  Valores Ausentes/Vazios por Coluna:")
        df_anomalia = pd.DataFrame(anomalias_colunas)
        print(df_anomalia.to_string(index=False))
    else:
        print("  Nenhum valor nulo ou texto em branco encontrado.")
        
    # 3. Validacao de Campos Numericos (Valores Negativos/Zerados)
    cols_numericas = df.select_dtypes(include=[np.number]).columns
    negativos_encontrados = []
    
    for col in cols_numericas:
        if col != pk and not col.endswith('_id'):
            negativos = (df[col] < 0).sum()
            zeros = (df[col] == 0).sum()
            if negativos > 0:
                negativos_encontrados.append(f"{col}: {negativos} valores negativos")
                
    if negativos_encontrados:
        print(f"  Valores Negativos: {', '.join(negativos_encontrados)}")
    else:
        print("  Valores numéricos dentro das faixas válidas (sem negativos indevidos).")


 TABELA: [BUYERS] | Linhas: 3001 | Colunas: 6
  Chaves Primárias ('id') duplicadas: 1
  Nenhum valor nulo ou texto em branco encontrado.
  Valores numéricos dentro das faixas válidas (sem negativos indevidos).

 TABELA: [ORDER_ITEMS] | Linhas: 214768 | Colunas: 6
  Sem duplicidades de registros ou de Chave Primária ('id').
  Nenhum valor nulo ou texto em branco encontrado.
  Valores numéricos dentro das faixas válidas (sem negativos indevidos).

 TABELA: [ORDERS] | Linhas: 80002 | Colunas: 6
  Chaves Primárias ('id') duplicadas: 2
  Nenhum valor nulo ou texto em branco encontrado.
  Valores numéricos dentro das faixas válidas (sem negativos indevidos).

 TABELA: [PAYMENTS] | Linhas: 80000 | Colunas: 6
  Sem duplicidades de registros ou de Chave Primária ('id').

  Valores Ausentes/Vazios por Coluna:
 Coluna  Nulos % Nulos  Vazios/Brancos
paid_at  11950  14.94%               0
  Valores numéricos dentro das faixas válidas (sem negativos indevidos).

 TABELA: [PRODUCTS] | Linhas: 800 | 

In [25]:
display(buyers.head())
display(order_items.head())
display(orders.head())
display(payments.head())
display(products.head())
display(sellers.head())


,id,name,city,state,segment,created_at
0,1,Nascimento & Costa Distribuidora Mercearia,Recife,DF,supermarket,2023-04-03 03:46:36
1,2,Lima & Almeida Atacado Mercearia,Rio de Janeiro,PE,convenience,2023-08-05 01:36:12
2,3,Ferreira & Costa Grupo Mercearia,São Paulo,PR,grocery,2023-07-10 11:37:01
3,4,Lima & Almeida Comércio Supermercado,Campo Grande,MT,convenience,2023-10-13 01:08:52
4,5,Ferreira & Ferreira Comércio Mercearia,São Paulo,PB,convenience,2024-04-20 00:23:27


,id,order_id,product_id,qty,unit_price,discount
0,1,1,500,6,240.06,117.48
1,2,1,778,6,126.47,94.54
2,3,2,346,39,492.08,98.86
3,4,2,248,20,278.55,588.33
4,5,3,701,9,298.75,272.33


,id,seller_id,buyer_id,status,created_at,total_value
0,1,114,2806,delivered,2023-08-22 05:25:59,1987.16
1,2,90,2586,processing,2024-07-19 03:57:54,24074.93
2,3,96,1849,completed,2024-06-14 22:16:15,2416.42
3,4,10,116,processing,2024-07-11 04:46:30,3871.48
4,5,16,2195,delivered,2023-09-07 20:06:40,15367.65


,id,order_id,paid_at,amount,method,status
0,1,1,2023-08-23 19:25:59,1987.16,transfer,paid
1,2,2,2024-07-20 17:57:54,24074.93,boleto,paid
2,3,3,2024-06-16 10:16:15,2416.42,transfer,paid
3,4,4,2024-07-11 11:46:30,3871.48,boleto,paid
4,5,5,2023-09-08 00:06:40,15367.65,transfer,paid


,id,name,category,seller_id,active,unit_cost
0,1,Produto Laticínios Linha 1,Snacks,31,1,104.39
1,2,Produto Grãos Linha 2,Carnes,71,1,153.19
2,3,Produto Enlatados Linha 3,Grãos,9,0,90.57
3,4,Produto Snacks Linha 4,Bebidas,13,1,158.39
4,5,Produto Carnes Linha 5,Grãos,58,1,124.43


,id,name,state,plan,created_at
0,1,Santos & Silva Atacado Distribuidora,BA,free,2023-04-17 00:16:11
1,2,Souza & Souza Comércio Distribuidora,DF,premium,2022-09-09 22:06:21
2,3,Santos & Almeida Distribuidora Distribuidora,AM,basic,2022-12-16 08:31:25
3,4,Nascimento & Ferreira Alimentos Distribuidora,GO,basic,2023-05-30 23:12:37
4,5,Silva & Santos Suprimentos Distribuidora,BA,premium,2023-03-06 10:29:33


In [26]:
# Desafio 1 
query_desafio1 = """
WITH orders_dedup AS (
    -- Deduplicação preventiva da Chave Primária (id)
    SELECT 
        id,
        MAX(total_value) AS total_value,
        MAX(status) AS status,
        MAX(created_at) AS created_at
    FROM orders
    GROUP BY id
),
orders_validos AS (
    SELECT 
        id AS order_id,
        total_value,
        status,
        created_at,
        strftime('%Y-%m', created_at) AS ano_mes
    FROM orders_dedup
    WHERE status IN ('completed', 'delivered')
),
max_data AS (
    SELECT MAX(DATE(created_at)) AS data_maxima FROM orders_validos
)
SELECT 
    ov.ano_mes AS mes,
    COUNT(DISTINCT ov.order_id) AS total_pedidos,
    ROUND(SUM(COALESCE(ov.total_value, 0)), 2) AS faturamento_bruto,
    ROUND(
        SUM(COALESCE(ov.total_value, 0)) / NULLIF(COUNT(DISTINCT ov.order_id), 0), 
        2
    ) AS ticket_medio
FROM orders_validos ov
CROSS JOIN max_data md
WHERE DATE(ov.created_at) >= DATE(md.data_maxima, '-12 months')
GROUP BY ov.ano_mes
ORDER BY mes DESC;
"""

resultado_desafio1 = pysqldf(query_desafio1)
display(resultado_desafio1)

,mes,total_pedidos,faturamento_bruto,ticket_medio
0,2024-11,3643,56710238.63,15566.91
1,2024-10,3863,62493439.13,16177.44
2,2024-09,3779,60274462.50,15949.84
3,2024-08,3743,60102539.82,16057.32
4,2024-07,3862,59769406.95,15476.28
5,2024-06,3644,58797774.37,16135.50
6,2024-05,3848,60730315.72,15782.31
7,2024-04,3653,57779312.59,15816.95
8,2024-03,3877,61525913.64,15869.46
9,2024-02,3628,57926376.26,15966.48


In [27]:
# Desafio 2
query_desafio2 = """
WITH orders_dedup AS (
    -- Deduplicação preventiva de orders
    SELECT 
        id,
        MAX(seller_id) AS seller_id,
        MAX(total_value) AS total_value,
        MAX(status) AS status,
        MAX(created_at) AS created_at
    FROM orders
    GROUP BY id
),
max_data AS (
    SELECT MAX(DATE(created_at)) AS data_maxima FROM orders_dedup WHERE status IN ('completed', 'delivered')
),
trimestres AS (
    SELECT 
        data_maxima,
        strftime('%Y', data_maxima) || '-Q' || ((cast(strftime('%m', data_maxima) as integer) + 2) / 3) AS q_atual,
        strftime('%Y', DATE(data_maxima, '-3 months')) || '-Q' || ((cast(strftime('%m', DATE(data_maxima, '-3 months')) as integer) + 2) / 3) AS q_anterior
    FROM max_data
),
orders_classificados AS (
    SELECT 
        o.id AS order_id,
        o.seller_id,
        o.total_value,
        strftime('%Y', o.created_at) || '-Q' || ((cast(strftime('%m', o.created_at) as integer) + 2) / 3) AS trimestre
    FROM orders_dedup o
    WHERE o.status IN ('completed', 'delivered')
),
metricas_sellers AS (
    SELECT 
        s.id AS seller_id,
        s.name AS nome_seller,
        s.state AS estado,
        
        COUNT(DISTINCT CASE WHEN oc.trimestre = t.q_anterior THEN oc.order_id END) AS pedidos_q_anterior,
        SUM(CASE WHEN oc.trimestre = t.q_anterior THEN COALESCE(oc.total_value, 0) ELSE 0 END) AS gmv_q_anterior,
        
        COUNT(DISTINCT CASE WHEN oc.trimestre = t.q_atual THEN oc.order_id END) AS pedidos_q_atual,
        SUM(CASE WHEN oc.trimestre = t.q_atual THEN COALESCE(oc.total_value, 0) ELSE 0 END) AS gmv_q_atual
    FROM sellers s
    CROSS JOIN trimestres t
    INNER JOIN orders_classificados oc ON s.id = oc.seller_id
    GROUP BY s.id, s.name, s.state
)
SELECT 
    nome_seller as nome_vendedor,
    estado,
    ROUND(gmv_q_anterior, 2) AS gmv_trimestre_anterior,
    ROUND(gmv_q_atual, 2) AS gmv_trimestre_atual,
    ROUND(
        ((gmv_q_atual - gmv_q_anterior) / NULLIF(gmv_q_anterior, 0)) * 100, 
        2
    ) AS pct_crescimento
FROM metricas_sellers
WHERE pedidos_q_anterior >= 50 
  AND pedidos_q_atual >= 50
ORDER BY pct_crescimento DESC
LIMIT 10;
"""

resultado_desafio2 = pysqldf(query_desafio2)
display(resultado_desafio2)

,nome_vendedor,estado,gmv_trimestre_anterior,gmv_trimestre_atual,pct_crescimento
0,Rodrigues & Almeida Atacado Distribuidora,MG,876302.72,856460.38,-2.26
1,Costa & Silva Alimentos Distribuidora,BA,1017817.89,976148.08,-4.09
2,Costa & Santos Suprimentos Distribuidora,DF,876787.95,807935.32,-7.85
3,Nascimento & Souza Alimentos Distribuidora,SP,1031166.40,933683.69,-9.45
4,Almeida & Almeida Alimentos Distribuidora,RS,1023319.44,923462.61,-9.76
5,Nascimento & Almeida Alimentos Distribuidora,SP,949398.56,847228.59,-10.76
6,Lima & Almeida Grupo Distribuidora,CE,1052368.65,938858.41,-10.79
7,Almeida & Souza Mercado Distribuidora,PR,1095325.17,973049.07,-11.16
8,Costa & Santos Atacado Distribuidora,MG,1106858.77,954140.51,-13.80
9,Almeida & Rodrigues Distribuidora Distribuidora,RJ,1102447.24,911827.61,-17.29


In [ ]:
# Desafio 3
query_desafio3 = """
WITH orders_dedup AS (
    -- Deduplicação preventiva de orders para não multiplicar a junção com order_items
    SELECT 
        id,
        MAX(seller_id) AS seller_id,
        MAX(status) AS status,
        MAX(created_at) AS created_at
    FROM orders
    GROUP BY id
)
SELECT 
    o.id AS pedido_id,  -- ID do pedido
    s.id AS vendedor_id,  -- ID do vendedor
    s.name AS vendedor_name, -- Nome do vendedor
    o.created_at AS data_pedido,  -- Data de criação do pedido
    ROUND(SUM(oi.qty * oi.unit_price), 2) AS valor_bruto_pedido,  -- Valor bruto do pedido (sem desconto)
    ROUND(SUM(oi.discount), 2) AS desconto_total,  -- Valor total de desconto aplicado no pedido
    ROUND(
        (SUM(oi.discount) / NULLIF(SUM(oi.qty * oi.unit_price), 0)) * 100, 
        2
    ) AS pct_desconto  -- Percentual de desconto aplicado no pedido
FROM orders_dedup o
INNER JOIN sellers s ON o.seller_id = s.id
INNER JOIN order_items oi ON o.id = oi.order_id
WHERE o.status != 'cancelled' -- Considerando apenas pedidos não cancelados
GROUP BY o.id, s.id, s.name, o.created_at
HAVING (SUM(oi.discount) / NULLIF(SUM(oi.qty * oi.unit_price), 0)) > 0.40  -- Filtrando pedidos com mais de 40% de desconto
ORDER BY pct_desconto DESC;
"""

resultado_desafio3 = pysqldf(query_desafio3)
display(resultado_desafio3)

,pedido_id,vendedor_id,vendedor_name,data_pedido,valor_bruto_pedido,desconto_total,pct_desconto
0,72401,113,Costa & Oliveira Atacado Distribuidora,2024-01-21 02:43:43,930.30,558.02,59.98
1,42513,13,Costa & Costa Atacado Distribuidora,2024-03-27 02:30:57,4082.60,2447.96,59.96
2,36119,13,Costa & Costa Atacado Distribuidora,2024-11-14 01:05:01,21030.24,12598.00,59.90
3,53223,13,Costa & Costa Atacado Distribuidora,2024-05-06 01:56:47,7970.40,4773.99,59.90
4,71165,99,Santos & Almeida Suprimentos Distribuidora,2024-11-09 02:32:22,13385.60,8017.88,59.90
...,...,...,...,...,...,...,...
896,38595,13,Costa & Costa Atacado Distribuidora,2023-08-21 04:52:02,27785.26,11124.91,40.04
897,64306,89,Oliveira & Santos Distribuidora Distribuidora,2024-10-12 21:22:52,6907.95,2765.81,40.04
898,32057,80,Souza & Ferreira Alimentos Distribuidora,2024-03-10 03:51:38,24658.56,9868.74,40.02
899,59315,76,Santos & Rodrigues Distribuidora Distribuidora,2023-08-10 22:52:09,21647.49,8661.37,40.01


In [29]:
# Desafio 4 - Query Oficial de Engenharia usando NOT EXISTS
query_desafio4 = """
WITH item_rankings AS (
    -- Ranqueia os itens de cada pedido pelo preço unitário decrescente
    SELECT 
        order_id,
        product_id,
        CAST(unit_price AS REAL) AS unit_price,
        CAST(qty AS INTEGER) AS qty,
        DENSE_RANK() OVER (
            PARTITION BY order_id 
            ORDER BY CAST(unit_price AS REAL) DESC
        ) AS rank_preco_no_pedido
    FROM order_items
    WHERE product_id IS NOT NULL
),
volume_produtos AS (
    -- Consolida o volume total de vendas por produto
    SELECT 
        p.id AS product_id,
        p.name AS product_name,
        p.category AS categoria,
        SUM(CAST(oi.qty AS INTEGER)) AS total_unidades_vendidas
    FROM products p
    INNER JOIN order_items oi ON p.id = oi.product_id
    GROUP BY p.id, p.name, p.category
)
SELECT 
    vp.product_id,
    vp.product_name,
    vp.categoria,
    vp.total_unidades_vendidas
FROM volume_produtos vp
WHERE vp.total_unidades_vendidas > 1000
  AND NOT EXISTS (
      -- Filtro de Exclusão: Garante que NUNCA existiu uma ocorrência desse produto como #1 no pedido
      SELECT 1 
      FROM item_rankings ir 
      WHERE ir.product_id = vp.product_id 
        AND ir.rank_preco_no_pedido = 1
  )
ORDER BY vp.total_unidades_vendidas DESC;
"""

resultado_desafio4 = pysqldf(query_desafio4)
display(resultado_desafio4)

,product_id,product_name,categoria,total_unidades_vendidas
